This notebook generates the Financial District hex plot. This code processes and visualizes business openings and closures in downtown San Francisco and aggregates point-level business activity by location and year into a hexogonal grid.

In [ ]:
import json
import re
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
hex_plot = gpd.read_parquet('../../data/processed/fidi_hex_open_close.parquet')

if hex_plot.crs is None:
    hex_plot = hex_plot.set_crs(epsg=4326)

hex_plot = hex_plot.to_crs(epsg=4326)

In [ ]:
hex_plot = hex_plot[
    hex_plot["event_year"].between(2019, 2025)
].copy()

years = list(range(2019, 2026))

In [ ]:
def round_geojson(gdf, ndigits=5):
    raw = gdf.to_json()
    rounded = re.sub(r'-?\d+\.\d+', lambda m: str(round(float(m.group()), ndigits)), raw)
    return json.loads(rounded)

geojson_hex = round_geojson(hex_plot)

years = sorted(hex_plot["event_year"].dropna().astype(int).unique())

shared_max = max(hex_plot["Opening"].max(), hex_plot["Closing"].max(), 1)

In [ ]:
boundary = hex_plot.dissolve()
center = boundary.to_crs(epsg=7131).centroid.to_crs(epsg=4326)

center_lat = center.y.iloc[0]
center_lon = center.x.iloc[0]

neighborhood = "Financial District"

print(hex_plot.head())

In [ ]:
start_year = years[0]

df = hex_plot[hex_plot["event_year"] == start_year]

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "mapbox"}, {"type": "mapbox"}]],
    subplot_titles=("Openings", "Closings"),
    horizontal_spacing=0.03
)

fig.add_trace(
    go.Choroplethmapbox(
        geojson=geojson_hex,
        locations=df["hex_id"],
        z=df["Opening"],
        featureidkey="properties.hex_id",
        colorscale="Blues",
        zmin=0,
        zmax=shared_max,
        marker_opacity=0.78,
        marker_line_width=0,
        colorbar=dict(title="Openings", x=0.46),
        hovertemplate="Openings: %{z}<extra></extra>"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Choroplethmapbox(
        geojson=geojson_hex,
        locations=df["hex_id"],
        z=df["Closing"],
        featureidkey="properties.hex_id",
        colorscale="Reds",
        zmin=0,
        zmax=shared_max,
        marker_opacity=0.78,
        marker_line_width=0,
        colorbar=dict(title="Closings", x=1.02),
        hovertemplate="Closings: %{z}<extra></extra>"
    ),
    row=1, col=2
)

fig.update_layout(coloraxis_showscale=True)

In [ ]:
fig.frames = [
    go.Frame(
        name=str(year),
        data=[
            go.Choroplethmapbox(
                locations=hex_plot[hex_plot["event_year"] == year]["hex_id"],
                z=hex_plot[hex_plot["event_year"] == year]["Opening"],
                colorscale="Blues",
                zmin=0, zmax=shared_max,
                marker_opacity=0.78, marker_line_width=0,
                showscale=False,
                hovertemplate="Openings: %{z}<extra></extra>"
            ),
            go.Choroplethmapbox(
                locations=hex_plot[hex_plot["event_year"] == year]["hex_id"],
                z=hex_plot[hex_plot["event_year"] == year]["Closing"],
                colorscale="Reds",
                zmin=0, zmax=shared_max,
                marker_opacity=0.78, marker_line_width=0,
                showscale=False,
                hovertemplate="Closings: %{z}<extra></extra>"
            )
        ],
        traces=[0, 1],
        layout=go.Layout(
            title_text=f"Business Openings and Closings Hex Grids in {neighborhood}, {year}"
        )
    )
    for year in years
]

In [ ]:
fig.update_layout(
    title=f"Business Openings and Closings Hex Grids in {neighborhood}, {start_year}",
    height=700,
    margin=dict(l=10, r=10, t=80, b=40),
    mapbox=dict(style="carto-positron", center=dict(lat=center_lat, lon=center_lon), zoom=12),
    mapbox2=dict(style="carto-positron", center=dict(lat=center_lat, lon=center_lon), zoom=12),
    updatemenus=[dict(
        type="buttons", showactive=False, x=0.05, y=0, xanchor="left", yanchor="top",
        buttons=[
            dict(label="Play", method="animate", args=[None, {"frame": {"duration": 900, "redraw": True}, "transition": {"duration": 300}, "fromcurrent": True, "mode": "immediate"}]),
            dict(label="Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ]
    )],
    sliders=[dict(
        active=0, x=0.15, y=0, len=0.75,
        currentvalue=dict(prefix="Year: ", visible=True),
        steps=[dict(label=str(year), method="animate", args=[[str(year)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}]) for year in years]
    )]
)
fig.update_traces(showscale=True)
for frame in fig.frames:
    for trace in frame.data:
        trace.showscale = True

fig.write_html('../../outputs/downtown_hex.html', include_plotlyjs='cdn')
fig.show()